# Notebook 05 Evaluasi Otomatis dan Plot

Notebook ini membaca hasil inference, menghitung skor retrieval, keyword coverage, semantic similarity, citation hit, membuat tabel evaluasi, dan menyimpan plot otomatis.

In [ ]:
# Jalankan bila package belum ada
# !pip install -q pandas numpy matplotlib sentence-transformers scikit-learn openpyxl


In [ ]:
import json, re, math
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

GROUND_TRUTH_CSV = Path("ground_truth_eval_20.csv")
RESULT_CSV = Path("eval_outputs/rag_inference_results.csv")
OUTPUT_DIR = Path("eval_outputs")
PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if not RESULT_CSV.exists():
    raise FileNotFoundError("Hasil inference belum ada. Jalankan notebook 04 sampai file eval_outputs/rag_inference_results.csv terbentuk.")

df = pd.read_csv(RESULT_CSV)
print(df.shape)
df.head(2)


In [ ]:
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).lower()).strip()


def split_semicolon(s: str) -> List[str]:
    return [x.strip() for x in str(s).split(";") if x.strip()]


def keyword_coverage(answer: str, keywords: str) -> float:
    ks = split_semicolon(keywords)
    if not ks:
        return 0.0
    a = norm(answer)
    hit = 0
    for k in ks:
        kk = norm(k)
        kk_digits = re.sub(r"[^0-9]", "", kk)
        if kk in a:
            hit += 1
        elif kk_digits and kk_digits in re.sub(r"[^0-9]", "", a):
            hit += 1
    return hit / len(ks)


def contains_any_blob(blob: str, values: List[str]) -> int:
    b = norm(blob)
    for v in values:
        vv = norm(v)
        if vv and vv in b:
            return 1
        compact_v = re.sub(r"[^0-9a-z]", "", vv)
        compact_b = re.sub(r"[^0-9a-z]", "", b)
        if compact_v and compact_v in compact_b:
            return 1
    return 0


def retrieval_law_hit(row) -> int:
    laws = split_semicolon(str(row.get("expected_law_numbers", "")).replace(",", ";"))
    return contains_any_blob(row.get("retrieved_references_text", ""), laws)


def retrieval_article_hit(row) -> int:
    arts = split_semicolon(str(row.get("expected_articles", "")).replace(",", ";"))
    return contains_any_blob(row.get("retrieved_references_text", ""), arts)


def answer_citation_hit(row) -> int:
    target = str(row.get("expected_citations", "")) + " " + str(row.get("expected_articles", "")) + " " + str(row.get("expected_law_numbers", ""))
    return contains_any_blob(row.get("answer", ""), split_semicolon(target.replace(",", ";")))


def parse_refs(js: str) -> List[Dict[str, Any]]:
    try:
        return json.loads(js)
    except Exception:
        return []


def topk_article_rank(row) -> int:
    arts = split_semicolon(str(row.get("expected_articles", "")).replace(",", ";"))
    refs = parse_refs(row.get("retrieved_references_json", "[]"))
    for r in refs:
        blob = " ".join([str(r.get("reference", "")), str(r.get("pasal_id", "")), str(r.get("source_file", ""))])
        if contains_any_blob(blob, arts):
            return int(r.get("rank", 999))
    return 999


df["keyword_coverage"] = df.apply(lambda r: keyword_coverage(r.get("answer", ""), r.get("expected_keywords", "")), axis=1)
df["retrieval_law_hit"] = df.apply(retrieval_law_hit, axis=1)
df["retrieval_article_hit"] = df.apply(retrieval_article_hit, axis=1)
df["answer_citation_hit"] = df.apply(answer_citation_hit, axis=1)
df["expected_article_rank"] = df.apply(topk_article_rank, axis=1)
df["article_hit_at_3"] = (df["expected_article_rank"] <= 3).astype(int)
df["article_hit_at_8"] = (df["expected_article_rank"] <= 8).astype(int)

df[["id", "keyword_coverage", "retrieval_law_hit", "retrieval_article_hit", "answer_citation_hit", "expected_article_rank"]].head()


In [ ]:
SEMANTIC_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
sem_model = SentenceTransformer(SEMANTIC_MODEL)

expected = df["expected_answer"].fillna("").tolist()
answers = df["answer"].fillna("").tolist()
emb_expected = sem_model.encode(expected, normalize_embeddings=True, show_progress_bar=True)
emb_answers = sem_model.encode(answers, normalize_embeddings=True, show_progress_bar=True)
df["semantic_similarity"] = [float(np.dot(a, b)) for a, b in zip(emb_answers, emb_expected)]
df[["id", "semantic_similarity"]].head()


In [ ]:
# Skor akhir dibuat sederhana agar bisa otomatis sekali jalan.
# Bobot bisa kamu ubah sesuai kebutuhan laporan.
weights = {
    "semantic_similarity": 0.30,
    "keyword_coverage": 0.25,
    "retrieval_article_hit": 0.20,
    "retrieval_law_hit": 0.10,
    "answer_citation_hit": 0.10,
    "article_hit_at_3": 0.05,
}

df["overall_score"] = sum(df[k] * w for k, w in weights.items())
df["quality_label"] = pd.cut(
    df["overall_score"],
    bins=[-0.01, 0.50, 0.70, 0.85, 1.01],
    labels=["poor", "fair", "good", "excellent"]
)

summary = pd.DataFrame({
    "metric": list(weights.keys()) + ["overall_score", "latency_seconds"],
    "mean": [df[k].mean() for k in weights.keys()] + [df["overall_score"].mean(), df["latency_seconds"].mean()],
    "min": [df[k].min() for k in weights.keys()] + [df["overall_score"].min(), df["latency_seconds"].min()],
    "max": [df[k].max() for k in weights.keys()] + [df["overall_score"].max(), df["latency_seconds"].max()],
})
summary


In [ ]:
eval_csv = OUTPUT_DIR / "evaluation_table.csv"
summary_csv = OUTPUT_DIR / "evaluation_summary.csv"
excel_path = OUTPUT_DIR / "evaluation_report.xlsx"

df.to_csv(eval_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="detail")
    summary.to_excel(writer, index=False, sheet_name="summary")
print("saved", eval_csv.resolve())
print("saved", summary_csv.resolve())
print("saved", excel_path.resolve())


In [ ]:
def save_fig(name: str):
    path = PLOT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print("saved", path.resolve())
    plt.show()

plt.figure(figsize=(12, 6))
plt.bar(df["id"], df["overall_score"])
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.title("Overall Score per Question")
plt.xlabel("Question ID")
plt.ylabel("Score")
save_fig("01_overall_score_per_question.png")

plt.figure(figsize=(8, 5))
plt.hist(df["overall_score"], bins=8)
plt.xlim(0, 1)
plt.title("Overall Score Distribution")
plt.xlabel("Score")
plt.ylabel("Count")
save_fig("02_overall_score_distribution.png")

metric_cols = ["semantic_similarity", "keyword_coverage", "retrieval_law_hit", "retrieval_article_hit", "answer_citation_hit", "article_hit_at_3"]
metric_means = df[metric_cols].mean().sort_values()
plt.figure(figsize=(10, 5))
plt.barh(metric_means.index, metric_means.values)
plt.xlim(0, 1)
plt.title("Average Metric Scores")
plt.xlabel("Mean Score")
save_fig("03_average_metric_scores.png")

topic_scores = df.groupby("topic")["overall_score"].mean().sort_values()
plt.figure(figsize=(10, 5))
plt.barh(topic_scores.index, topic_scores.values)
plt.xlim(0, 1)
plt.title("Average Score by Topic")
plt.xlabel("Mean Overall Score")
save_fig("04_score_by_topic.png")

plt.figure(figsize=(8, 5))
plt.scatter(df["latency_seconds"], df["overall_score"])
for _, r in df.iterrows():
    plt.text(r["latency_seconds"], r["overall_score"], r["id"], fontsize=8)
plt.ylim(0, 1)
plt.title("Latency versus Overall Score")
plt.xlabel("Latency seconds")
plt.ylabel("Overall Score")
save_fig("05_latency_vs_score.png")

heat = df.set_index("id")[metric_cols]
plt.figure(figsize=(10, 7))
plt.imshow(heat.values, aspect="auto")
plt.xticks(range(len(metric_cols)), metric_cols, rotation=45, ha="right")
plt.yticks(range(len(heat.index)), heat.index)
plt.colorbar(label="Score")
plt.title("Metric Heatmap per Question")
save_fig("06_metric_heatmap.png")

label_counts = df["quality_label"].value_counts().reindex(["poor", "fair", "good", "excellent"]).fillna(0)
plt.figure(figsize=(7, 5))
plt.bar(label_counts.index.astype(str), label_counts.values)
plt.title("Quality Label Count")
plt.xlabel("Quality")
plt.ylabel("Count")
save_fig("07_quality_label_count.png")

plt.figure(figsize=(10, 5))
rank_values = df["expected_article_rank"].replace(999, np.nan)
plt.bar(df["id"], rank_values.fillna(9))
plt.axhline(3, linestyle="--")
plt.xticks(rotation=45)
plt.title("Rank of Expected Article in Retrieved References")
plt.xlabel("Question ID")
plt.ylabel("Rank. Value 9 means not found")
save_fig("08_expected_article_rank.png")


In [ ]:
report = f"""# Laporan Evaluasi RAG

Jumlah soal: {len(df)}

Rata-rata overall score: {df['overall_score'].mean():.3f}

Rata-rata semantic similarity: {df['semantic_similarity'].mean():.3f}

Retrieval law hit rate: {df['retrieval_law_hit'].mean():.3f}

Retrieval article hit rate: {df['retrieval_article_hit'].mean():.3f}

Answer citation hit rate: {df['answer_citation_hit'].mean():.3f}

Rata-rata latency detik: {df['latency_seconds'].mean():.2f}

File penting:

1. eval_outputs/evaluation_table.csv
2. eval_outputs/evaluation_summary.csv
3. eval_outputs/evaluation_report.xlsx
4. eval_outputs/plots
"""
report_path = OUTPUT_DIR / "evaluation_report.md"
report_path.write_text(report, encoding="utf-8")
print(report)
print("saved", report_path.resolve())
